In [1]:
import yfinance as yf
import pandas as pd
tickers=pd.read_csv('../../data_collection/data/stock_index/nasdaq100.csv',index_col=0)
tickers = tickers.index.tolist()
data = yf.download(tickers, period="5y")['Close']

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  101 of 101 completed


In [2]:
#drop na by columns
data = data.dropna(axis=1)
data.head()

Ticker,AAPL,ADBE,ADI,ADP,ADSK,AEP,AMAT,AMD,AMGN,AMZN,...,TSLA,TTD,TTWO,TXN,VRSK,VRTX,WBD,WDAY,XEL,ZS
Date,,,,,,,,,,,,,,,,,,,,,
2020-07-09,93.078026,460.839996,113.565277,131.312088,248.199997,68.274544,60.712212,57.259998,215.375290,159.131500,...,92.952003,46.299999,150.759995,113.389633,167.904633,295.399994,19.250000,197.559998,54.103237,127.410004
2020-07-10,93.240875,466.200012,113.848755,132.609650,244.240005,70.047607,59.851585,55.880001,213.133026,160.000000,...,102.976669,46.198002,153.910004,113.476570,168.845032,294.450012,20.230000,195.880005,54.556660,125.489998
2020-07-13,92.810715,442.470001,107.219002,131.086823,231.860001,70.555389,58.818840,53.590000,213.988831,155.199997,...,99.804001,43.063000,146.820007,111.989990,167.565384,284.820007,20.520000,184.770004,54.898865,119.910004
2020-07-14,94.346596,433.779999,107.913986,131.086823,234.600006,71.046494,59.803780,54.720001,216.599106,154.199997,...,101.120003,43.203999,151.050003,114.658890,169.252136,293.829987,20.650000,185.679993,55.300961,122.430000
2020-07-15,94.995453,433.010010,107.813393,133.898163,235.580002,70.488792,59.383022,55.340000,216.787354,150.443497,...,103.067329,43.637001,151.279999,114.884926,170.037399,295.109985,21.780001,183.229996,54.924522,119.839996


In [3]:
returns = data.pct_change().dropna()

volatility = returns.std()
avg_return = returns.mean()


from sklearn.cluster import KMeans
import pandas as pd

features = pd.DataFrame({
    'volatility': volatility,
    'avg_return': avg_return
})

kmeans = KMeans(n_clusters=3)
features['cluster'] = kmeans.fit_predict(features)


In [4]:
from sklearn.cluster import KMeans
import pandas as pd

features = pd.DataFrame({
    'volatility': volatility,
    'avg_return': avg_return
})

kmeans = KMeans(n_clusters=4)
features['cluster'] = kmeans.fit_predict(features)

In [5]:
features

,volatility,avg_return,cluster
Ticker,,,
AAPL,0.018809,0.000825,3
ADBE,0.022752,0.000113,0
ADI,0.020485,0.000822,0
ADP,0.013767,0.000772,3
ADSK,0.022027,0.000434,0
...,...,...,...
VRTX,0.018105,0.000531,3
WBD,0.034195,0.000176,2
WDAY,0.024069,0.000444,0


In [6]:
#elbow method to find optimal number of clusters
import plotly.express as px
from sklearn.cluster import KMeans
inertia = []
n_samples = features.shape[0]
for i in range(1, n_samples + 1):
    kmeans = KMeans(n_clusters=i)
    kmeans.fit(features[['volatility', 'avg_return']])
    inertia.append(kmeans.inertia_)

fig = px.line(x=range(1, n_samples + 1), y=inertia, labels={'x': 'Number of Clusters', 'y': 'Inertia'})
fig.update_layout(title='Elbow Method for Optimal Clusters')
fig.show()

In [7]:
import plotly.express as px
#also show the ticker in the plot
features['Ticker'] = features.index
# Create a scatter plot with Plotly Express
fig = px.scatter(features, x='volatility', y='avg_return', color='cluster', text='Ticker',
                 title='Volatility vs Average Return Clustering',
                 labels={'volatility': 'Volatility', 'avg_return': 'Average Return'})
# Update layout for better readability
fig.update_traces(textposition='top center')
fig.update_layout(
    xaxis_title='Volatility',
    yaxis_title='Average Return',
    legend_title='Cluster'
)
fig.show()

In [8]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# สเกลข้อมูลก่อนทำ PCA
scaler = StandardScaler()
scaled_data = scaler.fit_transform(returns.T)  # T เพื่อให้หุ้นเป็น observations

# ทำ PCA
pca = PCA(n_components=2)  # ลดเหลือ 2 มิติ
pca_result = pca.fit_transform(scaled_data)

# ดูว่า Component ไหนอธิบายข้อมูลได้มากแค่ไหน
print("Explained variance ratio:", pca.explained_variance_ratio_)

Explained variance ratio: [0.14685249 0.07203637]


In [9]:
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'], index=returns.columns)

# สร้าง scatter plotly
fig = px.scatter(pca_df, x='PC1', y='PC2', text=pca_df.index,
                 title='PCA of Stock Returns',
                 labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'})
fig.update_traces(textposition='top center')
fig.update_layout(
    xaxis_title='Principal Component 1',
    yaxis_title='Principal Component 2'
)
fig.show()
